In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Maan lijiye ki aapne pichle saare classes yahan import kar liye hain:
# from dataset import BEN14KDataset
# from model import Antariksh_ViT_Encoders, DisjointMaskGenerator
# from predictor import CrossAttnPredictor
# from losses import AntarikshLoss

def train_antariksh_jepa():
    # 1. Hyperparameters
    batch_size = 32
    epochs = 5  # Hackathon ke liye pehle 5 epochs par test karenge
    learning_rate = 1e-4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 2. Load Data
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # 3. Initialize Models
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    mask_gen = DisjointMaskGenerator(num_patches=225, mask_ratio=0.5)
    criterion = AntarikshLoss(lambda_vicreg=0.1, lambda_hgr=0.1).to(device)

    # 4. Optimizer (Dono Encoders aur Predictor ko ek saath train karna hai)
    optimizer = optim.AdamW(list(encoders.parameters()) + list(predictor.parameters()), lr=learning_rate)

    # 5. The Training Loop
    print("\n🚀 Starting Antariksh-JEPA Training...")
    
    for epoch in range(epochs):
        encoders.train()
        predictor.train()
        running_loss = 0.0

        for batch_idx, (s1_imgs, s2_imgs) in enumerate(dataloader):
            # Data ko GPU/CPU par bhejo
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            current_batch_size = s1_imgs.size(0)

            # A. Zero Gradients
            optimizer.zero_grad()

            # B. Generate Disjoint Masks
            s2_mask, s1_mask = mask_gen.generate_masks(current_batch_size)
            
            # C. Forward Pass: Encoders
            # S1 Latent: (Batch, 226, 192), S2 Latent: (Batch, 226, 192)
            s1_latent, s2_latent = encoders(s1_imgs, s2_imgs)
            
            # D. Filter Visible S2 Patches & Target Hidden S1 Patches
            # Hum Global CLS token (index 0) ko hamesha visible maante hain
            num_hidden = int(0.5 * 225)
            
            # Note: Real implementation mein PyTorch ke `gather` ya boolean indexing ka 
            # use karke exact masked vectors nikalne padte hain. 
            # Abhi ke liye hum unhe slice kar rahe hain for simplicity in this prototype.
            s2_visible_context = s2_latent[:, :226-num_hidden, :] 
            actual_sar_hidden = s1_latent[:, 1:num_hidden+1, :] 

            # E. Forward Pass: Predictor
            predicted_sar_hidden = predictor(s2_visible_context, num_hidden)

            # F. Calculate Super-Loss
            loss, l2, vic, hgr = criterion(predicted_sar_hidden, actual_sar_hidden)

            # G. Backpropagation & Update Weights
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Print stats every 10 batches
            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | "
                      f"Total Loss: {loss.item():.4f} | L2: {l2.item():.4f} | "
                      f"VICReg: {vic.item():.4f} | HGRMC: {hgr.item():.4f}")
                
        print(f"✅ Epoch {epoch+1} Completed. Avg Loss: {running_loss/len(dataloader):.4f}\n")

if __name__ == "__main__":
    train_antariksh_jepa()

In [ ]:
import os
import torch
import torch.optim as optim
# (Baaki saare purane imports same rahenge...)

def train_antariksh_jepa_full():
    # 1. Hyperparameters for Long Training
    batch_size = 32
    epochs = 100  # Set to 50 or 100 for the full run
    learning_rate = 1e-4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Checkpoint Folder banayein
    save_dir = "saved_models"
    os.makedirs(save_dir, exist_ok=True)
    best_loss = float('inf') # Best loss ko track karne ke liye

    # 2. Load Data (Aapka purana Dataloader code)
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # 3. Initialize Models & Optimizers
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    mask_gen = DisjointMaskGenerator(num_patches=225, mask_ratio=0.5)
    criterion = AntarikshLoss(lambda_vicreg=0.1, lambda_hgr=0.1).to(device)

    optimizer = optim.AdamW(list(encoders.parameters()) + list(predictor.parameters()), lr=learning_rate)

    # --- OPTIONAL: Resume from Checkpoint ---
    # Agar training beech mein ruk jaye, toh isko True kar dijiye
    resume_training = False
    if resume_training and os.path.exists(os.path.join(save_dir, "last_checkpoint.pth")):
        print("Resuming training from last saved checkpoint...")
        checkpoint = torch.load(os.path.join(save_dir, "last_checkpoint.pth"))
        encoders.load_state_dict(checkpoint['encoders_state'])
        predictor.load_state_dict(checkpoint['predictor_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['best_loss']
    else:
        start_epoch = 0

    print(f"\n🚀 Starting Full Training from Epoch {start_epoch+1} to {epochs}...")
    
    for epoch in range(start_epoch, epochs):
        encoders.train()
        predictor.train()
        running_loss = 0.0

        for batch_idx, (s1_imgs, s2_imgs) in enumerate(dataloader):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            current_batch_size = s1_imgs.size(0)

            optimizer.zero_grad()
            s2_mask, s1_mask = mask_gen.generate_masks(current_batch_size)
            
            s1_latent, s2_latent = encoders(s1_imgs, s2_imgs)
            
            num_hidden = int(0.5 * 225)
            s2_visible_context = s2_latent[:, :226-num_hidden, :] 
            actual_sar_hidden = s1_latent[:, 1:num_hidden+1, :] 

            predicted_sar_hidden = predictor(s2_visible_context, num_hidden)

            loss, l2, vic, hgr = criterion(predicted_sar_hidden, actual_sar_hidden)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            if batch_idx % 50 == 0: # Print less frequently to save console memory
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | "
                      f"Total Loss: {loss.item():.4f} | HGRMC: {hgr.item():.4f}")
                
        # --- END OF EPOCH: SAVE CHECKPOINTS ---
        avg_loss = running_loss / len(dataloader)
        print(f"✅ Epoch {epoch+1} Completed. Avg Loss: {avg_loss:.4f}\n")

        # Create the checkpoint dictionary
        checkpoint = {
            'epoch': epoch,
            'encoders_state': encoders.state_dict(),
            'predictor_state': predictor.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_loss': best_loss
        }

        # 1. Hamesha last epoch save karo (in case system crashes)
        torch.save(checkpoint, os.path.join(save_dir, "last_checkpoint.pth"))

        # 2. Agar loss ab tak ka sabse best hai, toh ek special copy save karo!
        if avg_loss < best_loss:
            print(f"🌟 New Best Model Found! Loss improved from {best_loss:.4f} to {avg_loss:.4f}. Saving weights...")
            best_loss = avg_loss
            torch.save(checkpoint, os.path.join(save_dir, "antariksh_best_model.pth"))

if __name__ == "__main__":
    train_antariksh_jepa_full()